# Defining a metric before you compute it

Core stage, lesson C12. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** write a metric as numerator, denominator, unit and window, see how each choice moves the number, count people instead of retried rows, and pair a primary metric with its inputs and a guardrail.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, GREEN, PURPLE, GOLD, RED = "#0072b2", "#00a07a", "#9a6be0", "#ad9100", "#c0392b"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, GREEN, PURPLE, GOLD]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0; when the tag cannot be reached, from the copy in a clone of the repository.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"


def load(name, **kwargs):
    """Read one CSV file of app-events: from the release tag, else from the local clone."""
    try:
        return pd.read_csv(BASE + name, **kwargs)
    except Exception:
        return pd.read_csv(LOCAL + name, **kwargs)


users = load("users.csv")
events = load("events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])    # 13 MB: a few seconds
subs = load("subscriptions.csv", parse_dates=["started_on", "ended_on", "updated_at"])
assign = load("experiment_assignments.csv")

# Known answer from the dataset README: 21,841 people, 177,379 events, 3,840 subscription rows and 15,868 experiment rows.
assert (len(users), len(events), len(subs), len(assign)) == (21841, 177379, 3840, 15868)

# The time of a person's signup_complete event is their signup time.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["signup_ts"] = users["user_id"].map(signup["event_ts"])

# Bought: a purchase event within 14 days of the signup. People who signed up by June 16 have a full 14 days of follow-up (the data ends June 30).
purchases = events[events["event_name"] == "purchase"]
first = purchases.merge(users[["user_id", "signup_ts"]], on="user_id")
within14 = first[first["event_ts"] <= first["signup_ts"] + pd.Timedelta(days=14)]
users["bought14"] = users["user_id"].isin(within14["user_id"]).astype(int)
full = users[users["signup_date"] <= "2026-06-16"]
assert round(float(full["bought14"].mean()), 3) == 0.12      # the README: 12.0 percent

print(f"Data loaded: {len(users):,} people, {len(events):,} events; {100 * full['bought14'].mean():.1f} percent of the {len(full):,} people with a full follow-up bought Plus within 14 days.")

## A first look

In [ ]:
person = events[events["user_id"] == 10007].sort_values("event_ts").head(8)          # the first eight events of one person
for row in person.itertuples():
    print(row.user_id, row.event_ts, row.event_name, row.platform)

## Unit 1: A metric has four parts: the numerator

**Predict first.** "People active in June" is not a column. The cell counts the distinct people under four definitions of active. Which will be the largest, and will "any event" beat "a visit or an expense" by a few hundred people or by thousands?

In [ ]:
june = events[(events["event_ts"] >= "2026-06-01") & (events["event_ts"] < "2026-07-01")]     # June 1 to 30, by when it happened
definitions = {
    "any event": june,
    "a return visit (session_start)": june[june["event_name"] == "session_start"],
    "an expense added (add_expense)": june[june["event_name"] == "add_expense"],
    "a visit or an expense": june[june["event_name"].isin(["session_start", "add_expense"])],
}
active = {name: set(rows["user_id"]) for name, rows in definitions.items()}
for name, people in active.items():
    print(f"{len(people):>6,}  {name}")
extra = active["any event"] - active["a visit or an expense"]       # counted by the first definition and not by the last
print(len(extra))
shown = june[june["user_id"].isin(extra)].groupby("event_name")["user_id"].nunique().sort_values(ascending=False, kind="stable")
for name, people in shown.items():
    print(f"{people:>6,}  {name}")

## Unit 2: The window and the base

**Predict first.** The 5,077 people who signed up in May, read three ways: a cohort reading (a purchase within 14 days of their own signup), a reading with no window (a purchase any time up to June 30) and a calendar reading (purchases made in May, whoever signed up). Which reading will be the highest, and which the lowest? The cell then prints the 14-day conversion of two later groups of signups.

In [ ]:
may = users[(users["signup_date"] >= "2026-05-01") & (users["signup_date"] <= "2026-05-31")]     # the people who signed up in May
cohort = int(may["bought14"].sum())                                  # cohort reading: a purchase within 14 days of their own signup
ever = int(may["user_id"].isin(purchases["user_id"]).sum())          # no window: a purchase any time up to June 30
in_may = purchases[(purchases["event_ts"] >= "2026-05-01") & (purchases["event_ts"] < "2026-06-01")]     # calendar reading: purchases made in May
print(len(may), cohort, ever, len(in_may), int(in_may["user_id"].isin(may["user_id"]).sum()))
for label, count in (("cohort, 14 days", cohort), ("ever, to June 30", ever), ("calendar, May", len(in_may))):
    print(f"{100 * count / len(may):.1f}%  {label}")
late = users[users["signup_date"] >= "2026-06-17"]                   # signed up in the last two weeks of the data: their 14 days are not all over
june_early = users[(users["signup_date"] >= "2026-06-08") & (users["signup_date"] <= "2026-06-16")]     # June 8 to 16: a full 14 days
print(len(late), f"{100 * late['bought14'].mean():.1f}%", len(june_early), f"{100 * june_early['bought14'].mean():.1f}%")

**Predict first.** The cell computes the June activity rate (a visit or an expense) three times, out of three different bases. Which base will give the highest rate and which the lowest?

In [ ]:
june_active = active["a visit or an expense"] & set(users["user_id"])      # from unit 1, people who have an account row
existing = users[users["signup_date"] < "2026-06-01"]                      # accounts that already existed on June 1
may_events = events[(events["event_ts"] >= "2026-05-01") & (events["event_ts"] < "2026-06-01")]
may_active = set(may_events[may_events["event_name"].isin(["session_start", "add_expense"])]["user_id"]) & set(users["user_id"])
bases = {"all accounts": set(users["user_id"]), "accounts created before June 1": set(existing["user_id"]), "people active in May": may_active}
for label, base in bases.items():
    n = len(june_active & base)
    print(f"{len(base):>6,} {n:>6,} {100 * n / len(base):5.1f}%  {label}")

## Unit 3: The unit: rows or people

**Predict first.** A client retry sometimes wrote the same event twice. The cell finds the second copies and recounts the June `add_expense` events without them. Will the number of people who added an expense change? Will the number of rows?

In [ ]:
key = ["user_id", "event_ts", "event_name", "platform"]
retry = events.duplicated(subset=key)                                # True for the second copy of an event written twice
print(int(retry.sum()))
expenses = june[june["event_name"] == "add_expense"]
clean = expenses[~retry.loc[expenses.index]]                         # the same events without the second copies
print(len(expenses), len(clean), expenses["user_id"].nunique(), clean["user_id"].nunique())
print(f"{len(expenses) / expenses['user_id'].nunique():.2f} {len(clean) / clean['user_id'].nunique():.2f}")
print(len(purchases), purchases["user_id"].nunique())                # purchase rows against the people who bought

## Unit 4: A goal, its components and a guardrail

**Predict first.** The cell builds the subscriber table for January to June from the latest row of each subscription: signups, new subscriptions, cancellations and the people paying at the end of the month. Will the number paying grow every month? And will June's new subscriptions be above or below May's?

In [ ]:
latest = subs.sort_values("updated_at").groupby("subscription_id").tail(1)       # the current state of each of the 2,905 subscriptions
users["month"] = users["signup_date"].str[:7]
previous = 0
for end in ["2026-01-31", "2026-02-28", "2026-03-31", "2026-04-30", "2026-05-31", "2026-06-30"]:
    end = pd.Timestamp(end)
    start = end.replace(day=1)
    signups = int((users["month"] == f"{end.year}-{end.month:02d}").sum())
    new = int(latest["started_on"].between(start, end).sum())
    canceled = int(latest["ended_on"].between(start, end).sum())
    paying = int(((latest["started_on"] <= end) & (latest["ended_on"].isna() | (latest["ended_on"] > end))).sum())
    assert paying == previous + new - canceled                       # the stock moves by the flows
    previous = paying
    print(f"{end:%Y-%m}  signups {signups:>5,}  new {new:>4,}  canceled {canceled:>4,}  paying {paying:>5,}")

**Predict first.** `paywall_v2` lifted 14-day conversion. A guardrail for it: of the people who bought, how many canceled within 30 days of starting? Will the variant's buyers cancel more or less often than the control's buyers, and will the difference be clearly more than chance (p below 0.05)?

In [ ]:
pw = assign[(assign["experiment"] == "paywall_v2") & assign["converted"]]        # the buyers of the experiment
first_sub = latest.sort_values("started_on").groupby("user_id").head(1)           # each buyer's first subscription
buyers = pw.merge(first_sub, on="user_id", validate="one_to_one")
buyers["canceled30"] = buyers["ended_on"].notna() & ((buyers["ended_on"] - buyers["started_on"]).dt.days <= 30)
counts = buyers.groupby("variant")["canceled30"].agg(["sum", "count"])
for name in ["control", "variant"]:
    c, n = int(counts.loc[name, "sum"]), int(counts.loc[name, "count"])
    print(name, c, n, f"{100 * c / n:.1f}%")
table = pd.crosstab(buyers["variant"], buyers["canceled30"])
print(f"{stats.chi2_contingency(table, correction=False).pvalue:.3f}")
rates = counts["sum"] / counts["count"]
gap = rates["variant"] - rates["control"]                            # variant minus control
se = np.sqrt((rates * (1 - rates) / counts["count"]).sum())          # the standard error of the gap
print(f"{100 * gap:.1f} {100 * (gap - 1.96 * se):.1f} {100 * (gap + 1.96 * se):.1f}")

## Finish it

### Exercise 1: Finish it

**Task:** How many of the accounts that existed before May were active in May (a visit or an expense), and what share of those accounts is that? The two sets are given. Count the accounts in the first that are in the second, then the share in percent.

**Expected:** `answer_1` is a list of two numbers, the number of people and their share in percent rounded to 1 decimal.

In [ ]:
may_events = events[(events["event_ts"] >= "2026-05-01") & (events["event_ts"] < "2026-06-01")]            # given: May 1 to 31
active_may = set(may_events[may_events["event_name"].isin(["session_start", "add_expense"])]["user_id"])   # given: a visit or an expense in May
before_may = users[users["signup_date"] < "2026-05-01"]              # given: accounts created before May
# count how many of these accounts were active in May, then their share of the accounts in percent
answer_1 = None  # replace None with [people, percent]: percent rounded to 1 decimal

In [ ]:
_ = check("Exercise 1", "answer_1", [1916, 15.0], traps={(5526, 43.4): "That counts everyone active in May, including the people who signed up in May. Count only the accounts of before_may."})

<details>
<summary>Hint</summary>

Unit 2: `n = len(june_active & base)` counts the people in both sets, and `100 * n / len(base)` is the share. Here the base is `before_may` and the numerator set is `active_may`.

</details>

<details>
<summary>Solution</summary>

```python
people = int(before_may["user_id"].isin(active_may).sum())
answer_1 = [people, round(100 * people / len(before_may), 1)]
```

1,916 of the 12,739 accounts created before May were active in May: 15.0 percent. Counting everyone active in May gives 5,526 people (43.4 percent of those accounts), because the set also holds the people who signed up in May; the check cell reports it. Unit 2 shows 5,525 for the same people because it keeps only those who have a row in `users`: one person active in May belongs to a deleted account.

</details>

## On your own

### Exercise 2: On your own

**Task:** The conversion of the first half of April, read two ways. The cohort reading is the share of the people who signed up from April 1 to 15 who bought within 14 days of their own signup. The calendar reading is the number of purchases made from April 1 to 15, whoever made them, divided by the number of signups of those days. Before you start, say in one sentence what the result looks like: two percentages. Set `answer_2` to the list [cohort percent, calendar percent], each rounded to 1 decimal. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [12.3, 13.7], traps={(13.7, 12.3): "Those are the two readings in the other order. The cohort reading comes first."})

<details>
<summary>Hint</summary>

Unit 2 built both readings for May. Change the dates to April 1 to 15: the cohort reading uses the column `bought14` of the April signups, the calendar reading counts the rows of `purchases` made on those days.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Two percentages between 12 and 14, and the calendar reading is the larger."*

```python
april = users[(users["signup_date"] >= "2026-04-01") & (users["signup_date"] <= "2026-04-15")]
in_april = purchases[(purchases["event_ts"] >= "2026-04-01") & (purchases["event_ts"] < "2026-04-16")]
```

</details>

<details>
<summary>Solution</summary>

```python
april = users[(users["signup_date"] >= "2026-04-01") & (users["signup_date"] <= "2026-04-15")]
in_april = purchases[(purchases["event_ts"] >= "2026-04-01") & (purchases["event_ts"] < "2026-04-16")]
answer_2 = [round(float(100 * april["bought14"].mean()), 1), round(100 * len(in_april) / len(april), 1)]
```

218 of the 1,767 people who signed up from April 1 to 15 bought within 14 days of their own signup: 12.3 percent. The 242 purchases made on those days divided by the same 1,767 signups give 13.7 percent, but only 182 of those purchases were made by people who signed up in those days. The other 60 came from earlier signups, so the numerator and the denominator of the calendar reading describe different people.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague computes the 14-day conversion of the people who signed up from January 1 to 11 from the table `users` and gets a number a little higher than it should be. Fix the denominator so that it counts everyone who signed up in those days, and set `answer_3` to the conversion in percent rounded to 1 decimal.

**Expected:** `answer_3` is a number in percent, rounded to 1 decimal.

In [ ]:
january = users[users["signup_date"] <= "2026-01-11"]                # colleague: "everyone who signed up from January 1 to 11"
answer_3 = round(float(100 * january["bought14"].mean()), 1)         # colleague's conversion, in percent

In [ ]:
_ = check("Exercise 3", "answer_3", 14.4, traps={15.0: "That is the colleague's number: its denominator is the table of accounts that still exist. Count the people from the signup events instead."})

<details>
<summary>Hint</summary>

Where does the table `users` get its rows from, and could a person who signed up be missing from it? Count the `signup_complete` events of those days.

</details>

<details>
<summary>Solution</summary>

```python
signup_events = events[(events["event_name"] == "signup_complete") & (events["event_ts"] < "2026-01-12")]
people = signup_events["user_id"].nunique()
answer_3 = round(100 * int(january["bought14"].sum()) / people, 1)
```

The table `users` holds 948 of the 988 people who signed up from January 1 to 11, because 40 accounts were deleted and left no row, and none of them had bought (every purchase belongs to a person who is in `users`). 142 buyers out of 988 people is 14.4 percent, against 15.0 percent out of 948. The check cell reports 15.0 if you keep the colleague's denominator.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")